# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
# Display overview
print(metadata.name)
print(metadata.description)


## 2. Data Overview
Show available record sets, fields, and their `@id` values.

Each entity in Croissant (record sets, fields, columns) is referenced by its `@id`.

In [ ]:
# Discover record sets (tables) in the dataset
record_sets = metadata.recordSet
print("Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']} ({rs['name'] if 'name' in rs else 'Unnamed'})")

# For each record set, list fields and columns by their @id
for rs in record_sets:
    print(f"\nFields and columns for record set {rs['@id']}:")
    fields = rs.get('field', [])
    if fields:
        for f in fields:
            print(f"Field @id: {f['@id']} | name: {f.get('name','')} | dataType: {f.get('dataType','')}")
    columns = rs.get('column', [])
    if columns:
        for c in columns:
            print(f"Column @id: {c['@id']} | name: {c.get('name','')} | dataType: {c.get('dataType','')}")

## 3. Data Extraction
Load each record set as a Pandas DataFrame using `mlcroissant`.

Continue referencing record sets and fields by their `@id`.

In [ ]:
# Record sets @id list
record_set_ids = [rs['@id'] for rs in record_sets]

# Load each record set into a DataFrame
dfs = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)

# Show available columns for first record set
first_rs_id = record_set_ids[0]
print(f"Columns in record set '{first_rs_id}':", dfs[first_rs_id].columns.tolist())
dfs[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common EDA steps such as filtering on numeric fields, normalizing, and grouping.

#### Steps:
- Select a numeric field by its `@id`
- Filter records
- Normalize values
- Group by a categorical field

In [ ]:
# For demonstration, choose first numeric field from the first record set

# Find a numeric field @id
fields = record_sets[0].get('field', [])
numeric_field_id = None
for f in fields:
    if f.get('dataType','').lower() in ('float', 'integer', 'number'):
        numeric_field_id = f['@id']
        print(f"Selected numeric field: {numeric_field_id} ({f.get('name','')})")
        break
if numeric_field_id is None:
    # fallback: use first column if fields are not available
    numeric_field_id = dfs[first_rs_id].columns[0]

# Set threshold for filtering
threshold = 10
df = dfs[first_rs_id]
# Filter records if numeric
try:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for field {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
except Exception as e:
    print("Numeric EDA not possible, likely due to non-numeric data:", e)

# Grouping: choose a categorical field @id (if exists)
group_field_id = None
for f in fields:
    if f.get('dataType','').lower() == 'text':
        group_field_id = f['@id']
        print(f"\nSelected group field: {group_field_id} ({f.get('name','')})")
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped mean by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields, referencing fields and record sets by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of numeric field for the first record set
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].dropna().hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If grouping field exists, plot mean of numeric field by group
if group_field_id and group_field_id in df.columns:
    means = df.groupby(group_field_id)[numeric_field_id].mean()
    means.plot(kind='bar')
    plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to access and explore the FAIR^2 dataset using the `mlcroissant` library. Key steps included referencing all entities by their `@id`, loading record sets into dataframes, filtering and normalizing numeric fields, and visualizing distributions. For further analysis, consult the dataset schema and metadata for detailed descriptions and relationships between record sets.
